In [53]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
from IPython.display import display

In [54]:
data_path = Path.cwd().parent.joinpath("data")
gen_1 = os.path.join(data_path, "processed",  "gen1_clean.csv")
gen_2 = os.path.join(data_path, "processed",  "gen2_clean.csv")
gen_3 = os.path.join(data_path, "processed",  "gen3_clean.csv")


In [55]:
gen1 = pd.read_csv(gen_1)
gen2 = pd.read_csv(gen_2)
gen3 = pd.read_csv(gen_3)

# Rename sensor columns with generator prefix, drop date from gen2 and gen3
gen1 = gen1.rename(columns={c: f"gen1_{c}" for c in gen1.columns if c != 'date'})
gen2 = gen2.rename(columns={c: f"gen2_{c}" for c in gen2.columns if c != 'date'})
gen3 = gen3.rename(columns={c: f"gen3_{c}" for c in gen3.columns if c != 'date'})

# Stack sideways — one shared date column, all sensors as columns
df = pd.concat(
    [gen1, gen2.drop(columns=['date']), gen3.drop(columns=['date'])],
    axis=1  # axis=1 means sideways, not stacked
)

print(df.shape)          # should be (186, 34) — 1 date + 11*3 sensors
print(df.columns.tolist())
display(df.head(10))

(186, 34)
['date', 'gen1_load_MW', 'gen1_vent_pressure_bar', 'gen1_steam_flow_th', 'gen1_scrubber_temp_C', 'gen1_scrubber_pressure_bar', 'gen1_lh_inlet_temp_C', 'gen1_rh_inlet_temp_C', 'gen1_conductivity_uMHO', 'gen1_chest_pressure_barg', 'gen1_exhaust_pressure_bara', 'gen1_exhaust_temp_C', 'gen2_load_MW', 'gen2_vent_pressure_bar', 'gen2_steam_flow_th', 'gen2_scrubber_temp_C', 'gen2_scrubber_pressure_bar', 'gen2_lh_inlet_temp_C', 'gen2_rh_inlet_temp_C', 'gen2_conductivity_uMHO', 'gen2_chest_pressure_barg', 'gen2_exhaust_pressure_bara', 'gen2_exhaust_temp_C', 'gen3_load_MW', 'gen3_vent_pressure_bar', 'gen3_steam_flow_th', 'gen3_scrubber_temp_C', 'gen3_scrubber_pressure_bar', 'gen3_lh_inlet_temp_C', 'gen3_rh_inlet_temp_C', 'gen3_conductivity_uMHO', 'gen3_chest_pressure_barg', 'gen3_exhaust_pressure_bara', 'gen3_exhaust_temp_C']


,date,gen1_load_MW,gen1_vent_pressure_bar,gen1_steam_flow_th,gen1_scrubber_temp_C,gen1_scrubber_pressure_bar,gen1_lh_inlet_temp_C,gen1_rh_inlet_temp_C,gen1_conductivity_uMHO,gen1_chest_pressure_barg,...,gen3_vent_pressure_bar,gen3_steam_flow_th,gen3_scrubber_temp_C,gen3_scrubber_pressure_bar,gen3_lh_inlet_temp_C,gen3_rh_inlet_temp_C,gen3_conductivity_uMHO,gen3_chest_pressure_barg,gen3_exhaust_pressure_bara,gen3_exhaust_temp_C
0,2018-01-11,30.5,5.47,261.3,159.1,5.0,154.6,158.5,4.0,5.211,...,5.46,263.6,159.1,5.0,154.6,158.5,3.2,5.202,0.121,49.8
1,2018-02-11,30.4,5.51,264.3,159.1,5.0,154.7,158.6,3.0,5.248,...,5.49,217.2,159.5,5.0,155.4,159.3,6.8,4.096,0.095,44.7
2,2018-02-11,30.7,5.65,266.4,159.9,5.5,155.4,159.3,4.4,5.329,...,5.49,217.2,159.5,5.0,155.4,159.3,3.2,4.096,0.095,44.7
3,2018-03-11,25.7,5.66,237.3,150.4,5.6,156.1,159.9,3.8,4.662,...,5.69,259.2,150.3,5.6,156.1,160.0,3.8,4.657,0.140,52.4
4,2018-03-11,30.8,5.49,266.2,159.2,5.4,154.8,158.7,8.0,5.221,...,5.69,259.2,150.3,5.6,156.1,160.0,3.2,4.657,0.140,52.4
5,2018-04-11,25.7,5.67,241.3,160.2,5.6,156.0,159.0,3.4,4.600,...,5.67,241.3,160.2,5.6,156.0,159.0,3.4,4.602,0.137,52.2
6,2018-04-11,25.6,5.50,243.2,159.5,5.4,155.0,158.8,4.4,4.658,...,5.47,232.0,159.4,5.4,155.0,155.0,7.0,4.461,0.131,51.6
7,2018-05-11,30.8,5.47,260.7,158.0,5.3,154.4,158.3,3.0,5.200,...,5.47,251.9,158.9,5.3,154.6,158.4,3.4,5.053,0.111,48.0
8,2018-05-11,25.3,5.42,240.0,159.0,5.3,154.5,158.3,3.6,4.571,...,5.45,262.0,159.0,5.3,154.2,158.0,3.4,5.101,0.115,49.4
9,2018-06-11,30.6,5.53,263.5,159.5,5.4,154.7,158.7,3.0,5.235,...,5.53,265.8,158.9,5.4,154.6,158.6,3.0,5.189,0.114,48.7


In [56]:
# Save the dataset
raw_path = Path.cwd().parent / "data" / "processed"
df.to_csv(raw_path / "merged_olkaria_data.csv", index=False)

In [57]:
# Derived Physical Features
for gen in ['gen1', 'gen2', 'gen3']:
    # LH + RH are 0.92 correlated — compress into average (shared signal)
    # and asymmetry (the unique fault signal the EDA flagged)
    df[f'{gen}_inlet_temp_avg']      = (df[f'{gen}_lh_inlet_temp_C'] + df[f'{gen}_rh_inlet_temp_C']) / 2
    df[f'{gen}_inlet_temp_asymmetry'] = (df[f'{gen}_lh_inlet_temp_C'] - df[f'{gen}_rh_inlet_temp_C']).abs()
    # Pressure × Temp = proxy for exhaust energy density (coupled subsystem)
    df[f'{gen}_exhaust_enthalpy_proxy'] = df[f'{gen}_exhaust_pressure_bara'] * df[f'{gen}_exhaust_temp_C']

    # How much pressure is the turbine actually dropping across itself?
    df[f'{gen}_pressure_drop'] = df[f'{gen}_vent_pressure_bar'] - df[f'{gen}_exhaust_pressure_bara']

    # MW generated per tonne/hr of steam — efficiency ratio
    df[f'{gen}_steam_utilization'] = df[f'{gen}_load_MW'] / df[f'{gen}_steam_flow_th']

# Drop the raw redundant columns now that we've extracted what we need from them
cols_to_drop = [c for c in df.columns if c.endswith('_lh_inlet_temp_C') or c.endswith('_rh_inlet_temp_C')]
df = df.drop(columns=cols_to_drop)

# Drop raw exhaust columns — already compressed into exhaust_enthalpy_proxy
cols_to_drop = [c for c in df.columns
                if c.endswith('_exhaust_pressure_bara') or c.endswith('_exhaust_temp_C')]
df = df.drop(columns=cols_to_drop)

print(f"Shape after FE cell 1: {df.shape}")
print(df.columns.tolist())

Shape after FE cell 1: (186, 37)
['date', 'gen1_load_MW', 'gen1_vent_pressure_bar', 'gen1_steam_flow_th', 'gen1_scrubber_temp_C', 'gen1_scrubber_pressure_bar', 'gen1_conductivity_uMHO', 'gen1_chest_pressure_barg', 'gen2_load_MW', 'gen2_vent_pressure_bar', 'gen2_steam_flow_th', 'gen2_scrubber_temp_C', 'gen2_scrubber_pressure_bar', 'gen2_conductivity_uMHO', 'gen2_chest_pressure_barg', 'gen3_load_MW', 'gen3_vent_pressure_bar', 'gen3_steam_flow_th', 'gen3_scrubber_temp_C', 'gen3_scrubber_pressure_bar', 'gen3_conductivity_uMHO', 'gen3_chest_pressure_barg', 'gen1_inlet_temp_avg', 'gen1_inlet_temp_asymmetry', 'gen1_exhaust_enthalpy_proxy', 'gen1_pressure_drop', 'gen1_steam_utilization', 'gen2_inlet_temp_avg', 'gen2_inlet_temp_asymmetry', 'gen2_exhaust_enthalpy_proxy', 'gen2_pressure_drop', 'gen2_steam_utilization', 'gen3_inlet_temp_avg', 'gen3_inlet_temp_asymmetry', 'gen3_exhaust_enthalpy_proxy', 'gen3_pressure_drop', 'gen3_steam_utilization']


In [59]:
# Temporal & Cross-Generator Features
df['date'] = pd.to_datetime(df['date'])
# --- REGIME FLAG ---
# EDA showed a permanent structural break in conductivity across all three generators
# between Nov 2018 and Jan 2019. Pre-break = stable low-conductivity steam conditions.
# Post-break = sustained contamination-prone regime with frequent critical events.
# Encoding this as a binary flag lets the model treat the two eras differently
# rather than assuming stationarity across the full 6-year period.
df['post_regime_shift'] = (df['date'] >= '2019-01-01').astype(int)

# --- TEMPORAL FEATURES ---
# Geothermal reservoirs are not fully immune to seasonal effects — rainfall patterns,
# reinjection cycles, and maintenance windows tend to cluster by month.
# EDA also showed year-on-year reservoir drift (declining capacity factors).
# Month captures intra-year seasonality; year captures long-term degradation trend.
df['month'] = df['date'].dt.month
df['year']  = df['date'].dt.year

# CROSS-GENERATOR LOAD DIVERGENCE
# All three generators draw from the same Olkaria II wellfield but their readings
# are staggered by 4 hours (Gen1 @ 08:00, Gen2 @ 12:00, Gen3 @ 04:00).
# If the reservoir were perfectly stable, all three would read similar loads on the
# same date. Divergence between them on the same date therefore directly reflects
# intra-day reservoir drift, wellhead pressure drops, enthalpy decay, and
# shifting steam-to-brine ratios that cause off-design output drops.
# This validates the off-design operating conditions the EDA quantified (~83% CF).
df['load_divergence_g1_g2'] = (df['gen1_load_MW'] - df['gen2_load_MW']).abs()
df['load_divergence_g1_g3'] = (df['gen1_load_MW'] - df['gen3_load_MW']).abs()
df['load_divergence_g2_g3'] = (df['gen2_load_MW'] - df['gen3_load_MW']).abs()

# Single severity score — the worst divergence seen across all three on a given date.
# A high value on this column means the reservoir was in an unstable state that day.
df['max_load_divergence'] = df[['load_divergence_g1_g2',
                                 'load_divergence_g1_g3',
                                 'load_divergence_g2_g3']].max(axis=1)

# --- CROSS-GENERATOR STEAM DIVERGENCE ---
# Same logic applied to steam flow. Enthalpy decay in the reservoir shows up as
# declining steam mass flow before it shows up as MW drop — so steam divergence
# is an earlier upstream signal of the same reservoir drift phenomenon.
df['steam_divergence_g1_g2'] = (df['gen1_steam_flow_th'] - df['gen2_steam_flow_th']).abs()
df['steam_divergence_g1_g3'] = (df['gen1_steam_flow_th'] - df['gen3_steam_flow_th']).abs()
df['steam_divergence_g2_g3'] = (df['gen2_steam_flow_th'] - df['gen3_steam_flow_th']).abs()

print(f"Shape after FE cell 2: {df.shape}")
display(df[['date', 'post_regime_shift', 'month', 'year',
          'load_divergence_g1_g2', 'load_divergence_g1_g3',
          'max_load_divergence']].head(8))

Shape after FE cell 2: (186, 47)


,date,post_regime_shift,month,year,load_divergence_g1_g2,load_divergence_g1_g3,max_load_divergence
0,2018-01-11,0,1,2018,0.4,0.2,0.4
1,2018-02-11,0,2,2018,5.1,4.9,5.1
2,2018-02-11,0,2,2018,0.0,5.2,5.2
3,2018-03-11,0,3,2018,0.0,0.0,0.0
4,2018-03-11,0,3,2018,5.8,5.1,5.8
5,2018-04-11,0,4,2018,0.1,0.1,0.1
6,2018-04-11,0,4,2018,1.1,0.2,1.1
7,2018-05-11,0,5,2018,0.1,0.5,0.6
